# 02_价值形式拓扑 (Value Form Topology)

本 Notebook 验证**价值形式涌现**机制：
1. 物物交换 → 扩大价值形式 → 一般价值形式 → 货币形式
2. 阻抗路由网络如何决定一般等价物
3. 一般等价物入度 > 80% 的涌现

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from src.model.model import CapitalModel
from src.engine.value_form_router import ImpedanceRouter
from src.model.social_stage import SocialStage

## 1. 运行模拟直到部落阶段

In [ ]:
# 创建模型并运行到部落阶段
model = CapitalModel(
    width=100,
    height=100,
    num_foragers=30,
    num_tribe_members=50,
    seed=42
)

# 运行直到出现物物交换
for i in range(200):
    model.step()
    if i % 20 == 0:
        print(f"Step {i}: Stage={model.social_stage.value}, Population={model.get_population_count()}")
    
    # 检查是否出现商品交换
    if hasattr(model, 'landscape'):
        break

print(f"\nFinal Stage: {model.social_stage.value}")
print(f"Total Population: {model.get_population_count()}")

Step 0: Stage=primitive_horde, Population=80

Final Stage: primitive_horde
Total Population: 80


c:\Users\koltf\AppData\Local\Programs\Python\Python313\Lib\site-packages\mesa\mesa_logging.py:112: FutureWarning: The use of the `seed` keyword argument is deprecated, use `rng` instead. No functional changes.
  res = func(*args, **kwargs)


## 2. 价值形式演进分析

In [ ]:
# 创建阻抗路由器
router = ImpedanceRouter(model.social_graph)

# 分析交换历史
exchange_data = []
for agent in model._agent_lookup.values():
    for commodity in agent.commodity_inventory:
        if hasattr(commodity, 'exchange_status') and commodity.exchange_status == 'Exchanged':
            exchange_data.append({
                'agent_id': agent.unique_id,
                'commodity_type': commodity.physical_props.get('name', 'unknown'),
                'state': commodity.state.value
            })

df_exchanges = pd.DataFrame(exchange_data)
print(f"Total exchanges recorded: {len(df_exchanges)}")
if len(df_exchanges) > 0:
    print("\nCommodity types in exchange:")
    print(df_exchanges['commodity_type'].value_counts())

Total exchanges recorded: 0


## 3. 社会关系图拓扑分析

In [ ]:
# 分析社会关系图的拓扑结构
G = model.social_graph.graph

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.4f}")

# 计算入度（一般等价物的候选）
in_degrees = dict(G.in_degree())
top_receivers = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 nodes by in-degree (potential universal equivalent):")
for node_id, degree in top_receivers:
    print(f"  Node {node_id}: in-degree = {degree}")

Nodes: 80
Edges: 22
Density: 0.0035

Top 5 nodes by in-degree (potential universal equivalent):
  Node 41: in-degree = 2
  Node 43: in-degree = 2
  Node 78: in-degree = 2
  Node 37: in-degree = 1
  Node 38: in-degree = 1


## 4. 验证：一般等价物入度 > 80%

In [ ]:
# 检查是否存在入度 > 80% 的节点
if G.number_of_nodes() > 0:
    max_in_degree = max(in_degrees.values())
    total_nodes = G.number_of_nodes()
    
    if max_in_degree > 0:
        concentration_ratio = max_in_degree / total_nodes
        print(f"Max in-degree concentration: {concentration_ratio:.2%}")
        
        if concentration_ratio > 0.8:
            print("✓ 一般等价物涌现验证通过 (入度 > 80%)")
        else:
            print("○ 一般等价物尚未涌现 (需要更多交换)")
    else:
        print("○ 尚无入边，交换尚未开始")

Max in-degree concentration: 2.50%
○ 一般等价物尚未涌现 (需要更多交换)


## 5. 价值形式阶段判定

In [ ]:
# 使用阻抗路由器判定价值形式阶段
value_form_stage = router.determine_value_form_stage()
print(f"Current value form stage: {value_form_stage}")

if value_form_stage == "barter":
    print("Phase: 物物交换 (Barter) - 直接的双边交换")
elif value_form_stage == "expanded":
    print("Phase: 扩大价值形式 (Expanded) - 多方交换")
elif value_form_stage == "universal":
    print("Phase: 一般价值形式 (Universal) - 一般等价物出现")
elif value_form_stage == "money":
    print("Phase: 货币形式 (Money) - 固定的一般等价物")

print(f"\nUniversal equivalent candidate: {router.universal_equivalent}")

Current value form stage: barter
Phase: 物物交换 (Barter) - 直接的双边交换

Universal equivalent candidate: None


---

## 验收标准

根据开发大纲 M3:
- **M3**: 运行 500 步后某节点入度 > 80%，未经预设

本 Notebook 验证价值形式拓扑算法是否能在模拟中涌现出货币形式。